# Analiza streszczeń literatury młodziezowej

## Import bibliotek

In [76]:
import os
import json
import nltk
import string
import morfeusz2
import matplotlib as mlp
import numpy as np
import pandas as pd
import operator as op
import itertools as it
from wordcloud import WordCloud
from matplotlib import pyplot as plt
from matplotlib.colors import to_hex
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
from nltk.corpus import PlaintextCorpusReader
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import load_iris
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [ ]:
nltk.download('punkt_tab')
nltk.pathsec.ENFORCE = False

## Zdefiniowanie funkcji pomocniczych

In [ ]:
def print_dict(obj):
    print(json.dumps(obj, indent=4, ensure_ascii=False))

In [ ]:
def create_dir(path):
    if not os.path.exists(path):
        os.mkdir(path)

## Utworzenie korpusu dokumentów

In [ ]:
corpus_dir = "./Literature"
corpus = PlaintextCorpusReader(corpus_dir, r".*\.txt")
files_names = corpus.fileids()

In [ ]:
documents_dict = {}
for file_name in files_names:
    documents_dict[file_name] = corpus.raw(file_name)
print_dict(documents_dict)

In [ ]:
lengths = {}
for file_name in documents_dict:
    lengths[file_name] = {
        "pre": len(word_tokenize(documents_dict[file_name])),
    }
print_dict(lengths)

## Utworzenie macierzy częstości

In [ ]:
documents_df_pre = pd.DataFrame.from_dict(documents_dict, orient='index', columns=['content'])

In [ ]:
matrix_tf = CountVectorizer().fit_transform(documents_df_pre['content'])

In [ ]:
sparsity_tf = (1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))) * 100

In [ ]:
matrices = {
    "pre": {
        "words": matrix_tf.shape[1],
        "sparcity": sparsity_tf
    }
}

## Wstępne przygotowanie dokumentów

In [ ]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding="utf-8")
stopwords = stopwords_file.read().splitlines()
stopwords = stopwords[4:]  # Skip the first four lines of the file
stopwords_file.close()
stopwords

In [41]:
def lemmatize(text):
    morfer = morfeusz2.Morfeusz()
    segments = it.groupby(morfer.analyse(text), op.itemgetter(0,1))
    def disambiguate(group):
        pairs = ((len(descr), lemma) for _, _, (_, lemma, descr, _, _, ) in group)
        perpl, lemma = min(pairs)
        return lemma.split(":")[0] if lemma else None
    lemmas = (disambiguate(group) for _, group in segments)
    return " ".join(filter(str.isalpha, lemmas))

In [42]:
for file_name in documents_dict:
    documents_dict[file_name] = documents_dict[file_name].lower()
    documents_dict[file_name] = "".join([char for char in documents_dict[file_name] if char not in string.punctuation])
    documents_dict[file_name] = lemmatize(documents_dict[file_name])
    documents_dict[file_name] = " ".join([word for word in word_tokenize(documents_dict[file_name], language='polish') if word not in stopwords])
print_dict(documents_dict)

{
    "Harry Potter i Czara Ognia.txt": "lord voldemort wraz sługa glizdogonem przybywać dom należący niegdyś rodzina riddleów położony little hangleton układać plan dotyczyć zamordować harryego pottera mieć pomóc czarny odzyskać dawny moc mowa wierna sługa czarny wkrótce mieć znaleźć hogwarcie słyszeć mieszkać nieopodal ogrodnik frank bryka voldemort odkrywać mężczyzna podsłuchiwać zapraszać środek uśmiercać Harry budzić przerażający sen blizna pulsować okropny boleć próbować siebie przypomnieć czym śnić a pamięć wyłaniać obraza mroczny pokój wąż leżący kominek glizdogona lord voldemorta planować morderstwo Harry zastanawiać powinien ktoś powiedzieć problem blizna boleć ostatni razem voldemort wedrzeć hogwartu ostatecznie postanawiać napisać Syriusz zejść śniadanie trakt śniadanie wuj vernon otrzymywać lista weasleyowie oferować zabrać harryego finał mistrzostwo świat quidditchu ugościć reszta wakacje długi namysł użyć harryego odpowiedni argument wuj zgadzać wyjazd pokój Harry zastaw

In [43]:
for file_name in documents_dict:
    lengths[file_name]["post"] = len(word_tokenize(documents_dict[file_name]))
print_dict(lengths)

KeyError: 'Harry Potter i Czara Ognia.txt'

In [ ]:
lengths = pd.DataFrame.from_dict(lengths, orient='index')
lengths['change'] = lengths['post'] - lengths['pre']
lengths['change_percentage'] = ((lengths['change'] / lengths['pre']) * 100).round(2)
display(lengths)

## Utworzenie macierzy częstości

In [ ]:
documents_df_post = pd.DataFrame.from_dict(documents_dict, orient='index', columns=['content'])

In [ ]:
cv = CountVectorizer()
matrix_tf = cv.fit_transform(documents_df_post['content'])
tv = TfidfVectorizer()
matrix_tfidf = tv.fit_transform(documents_df_post['content'])

In [ ]:
sparsity_tf = (1 -(matrix_tf.nnz / matrix_tf.shape[0] * matrix_tf.shape[1])) * 100

In [ ]:
matrices["post"] = {
    "words": matrix_tf.shape[1],
    "sparsity": sparsity_tf
}

In [ ]:
matrices = pd.DataFrame.from_dict(matrices, orient="index")
display(matrices)

## Przygotowanie katalogów wyjściowych

In [ ]:
create_dir("./wordclouds")
create_dir("./wordclouds/pre")
create_dir("./wordclouds/post")
create_dir("./topic_models")
create_dir("./clustering")
create_dir("./ngrams")

## Chmury tagów

In [ ]:
wordcloud = WordCloud(
    background_color="white",
    max_words=5000,
    contour_width=3,
    contour_color="steel_blue"
)

In [ ]:
for index, row in documents_df_pre.iterrows():
    wordcloud.generate(row["content"])
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(index.replace(".txt", ""))
    plt.savefig(f"./wordclouds/pre/{index.replace(".txt", ".png")}")
    plt.close()

In [ ]:
for index, row in documents_df_post.iterrows():
    wordcloud.generate(row["content"])
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(index.replace(".txt", ""))
    plt.savefig(f"./wordclouds/post/{index.replace(".txt", ".png")}")
    plt.close()

## Analiza tematyk

In [ ]:
def plot_top_words(model, feature_names, n_top_words, title, subplots, colors):
    fig, axes = plt.subplots(*subplots, figsize=(30, 15), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[-n_top_words:]
        top_features = feature_names[top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7, color=colors[topic_idx])
        ax.set_title(f"Temat {topic_idx + 1}", fontdict={"fontsize": 30})
        ax.tick_params(axis="both", which="major", labelsize=20)
        for i in "top right left".split():
            ax.spines[i].set_visible(False)
        fig.suptitle(title, fontsize=40)

    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)
    plt.savefig("./topic_models/" + str(n_components) + "/topics/" + title.replace(" ", "_") + ".png")
    plt.close()

In [ ]:
def plot_documents(model, matrix, n_components, title, colors):
    documents_topics = pd.DataFrame(model.transform(matrix))
    documents_topics.columns = ["Temat " + str(i) for i in range(1, n_components+1)]
    documents_topics.index = [file_name.replace(".txt", "") for file_name in files_names]
    n_docs = len(documents_topics)
    plt.figure(figsize=(7, 4))
    left = [0] * n_docs
    for i, col in enumerate(documents_topics.columns):
        plt.barh(documents_topics.index, documents_topics[col], left=left, label=col, color=colors[i])
        left = [left[j]+ documents_topics[col].iloc[j] for j in range(n_docs)]
    plt.tight_layout()
    plt.savefig("./topic_models/" + str(n_components) + "/documents/" + title.replace(" ", "_") + ".png")
    plt.close()

In [ ]:
colors = ["plum", "pink", "powderblue", "lemonchiffon", "peachpuff",
            "lavender", "paleturquoise", "lightpink", "mediumorchid", "darkseagreen"]
feature_names =  cv.get_feature_names_out()
n_top_features = 20
size = [1,4]
n_components = size[0]*size[1]

In [ ]:
create_dir("./topic_models/topics"+str(n_components))
create_dir("./topic_models/topics"+str(n_components)+"/topics")
create_dir("./topic_models/topics"+str(n_components)+"/documents")

In [ ]:
lda = LatentDirichletAllocation(
    n_components=n_components,
    max_iter=5,
    learning_method="batch",
    learning_offset=50,
    random_state=0
)
lda.fit(matrix_tf)
plot_top_words(lda, feature_names, n_top_features, "Tematu w modeli LDA", size, colors)
plot_documents(lda, matrix_tf, n_components, "Tematu w modeli LDA", colors)

In [ ]:
nmf_fn = NMF(
    n_components=n_components,
    random_state=1,
    alpha_H=.00005,
    alpha_W=.0005,
    l1_ratio=.5
)
nmf_fn.fit(matrix_tf)
plot_top_words(nmf_fn, feature_names, n_top_features, "Tematy w modeli NMF", size, colors)
plot_documents(nmf_fn, matrix_tf, n_components, "Tematy w modeli NMF", colors)

In [ ]:
nmf_kl = NMF(
    n_components=n_components,
    random_state=1,
    alpha_H=.00005,
    alpha_W=.0005,
    l1_ratio=.5
)
nmf_kl.fit(matrix_tf)
plot_top_words(nmf_kl, feature_names, n_top_features, "Tematy w modeli NMF", size, colors)
plot_documents(nmf_kl, matrix_tf, n_components, "Tematy w modeli NMF", colors)

## Analiza skupień


In [79]:
def plot_dendrogram(model, labels, title, orientation="top"):
    groups = model.labels_
    n_samples = len(groups)
    n_merges = len(model.children_)
    counts = np.zeros(n_merges, dtype=int)
    group_sizes = np.bincount(groups)

    # Numer grupy każdego węzła. -1 oznacza połączenie różnych grup.
    node_groups = np.full(n_samples + n_merges, -1, dtype=int)
    node_groups[:n_samples] = groups
    leaf_merge_distances = np.zeros(n_samples)
    palette = [mlp.colors.to_hex(color) for color in plt.get_cmap("tab10").colors]

    for i, (left, right) in enumerate(model.children_):
        for child in (left, right):
            if child < n_samples:
                counts[i] += 1
                leaf_merge_distances[child] = model.distances_[i]
            else:
                counts[i] += counts[child - n_samples]
        if node_groups[left] == node_groups[right]:
            node_groups[n_samples + i] = node_groups[left]

    linkage_matrix = np.column_stack(
        [model.children_, model.distances_, counts]
    ).astype(float)

    def branch_color(node_id):
        group = node_groups[node_id]
        return palette[group] if group >= 0 else "#808080"

    plt.figure(figsize=(6,4))
    ax = plt.gca()
    tree = dendrogram(
        linkage_matrix, labels=list(labels), orientation=orientation,
        link_color_func=branch_color, ax=ax,
    )
    horizontal = orientation in ("left", "right")
    tick_labels = ax.get_yticklabels() if horizontal else ax.get_xticklabels()

    for tick, leaf in zip(tick_labels, tree["leaves"]):
        group = groups[leaf]
        color = palette[group]
        tick.set_color(color)

        # SciPy rysuje całe połączenie w jednym kolorze.
        # Pokoloruj osobno odcinek od liścia do jego pierwszego połączenia.
        if group_sizes[group] == 1:
            position = tick.get_position()[1 if horizontal else 0]
            distance = leaf_merge_distances[leaf]
            distances, positions = [0, distance], [position, position]
            x, y = (distances, positions) if horizontal else (positions, distances)
            ax.plot(
                x, y, color=color, zorder=3, scalex=False, scaley=False,
                marker="o" if distance == 0 else None, markevery=[0],
                markersize=4, clip_on=False,
            )
            
    plt.tight_layout()
    plt.savefig(f"./clustering/{title}.png")
    plt.close()

In [72]:
ed = euclidean_distances(
    matrix_tf,
    matrix_tf
).flatten().reshape(
    matrix_tf.shape[0],
    matrix_tf.shape[0]
)

In [73]:
clustering_ed_ward = AgglomerativeClustering(
    n_clusters = 6,
    metric = "euclidean",
    linkage = "ward",
    compute_distances = True
)
clustering_ed_ward.fit(ed)
print(clustering_ed_ward.labels_)
plot_dendrogram(
    clustering_ed_ward,
    labels = files_names,
    title = "euclidean_ward"
    orientation = "right"
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2396725216.py, line 12)

In [ ]:
cs = cosine_similarity(
    matrix_tf,
    matrix_tf
).flatten().reshape(
    matrix_tf.shape[0],
    matrix_tf.shape[0]
)

In [ ]:
clustering_cs_complete = AgglomerativeClustering(
    n_clusters = 6,
    metric = "cosine",
    linkage = "complete",
    compute_distances = True
)
clustering_cs_complete.fit(cs)
print(clustering_cs_complete.labels_)
plot_dendrogram(
    clustering_ed_ward,
    labels = files_names,
    title = "cosine_complete"
    orientation = "right"
)

## N-gramy

In [83]:
for n in range(1,4):
    create_dir("./ngrams/"+str(n))
    documents_tockenized = {}
    for key in documents_dict:
        documents_tockenized[key] = word_tokenize(documents_dict[key],"polish")
    for key in documents_tockenized:
        n_grams = pd.Series(ngrams(documents_tockenized[key], n)).value_counts()
        n_grams[:5].plot.barh()
        plt.title(key.replace(".txt", ""))
        plt.tight_layout()
        plt.savefig(f"./ngrams/{n}{key.replace(".txt", ".png")}")
        plt.close()

In [84]:
texts = word_tokenize(" ".join(documents_df_post["content"]), "polish") 
names = ["unigray", "bigramy", "trigramy"]
for n in range(1,4):
    n_grams = pd.Series(ngrams(texts, n)).value_counts()
    n_grams = pd.Series(ngrams(documents_tockenized[key], n)).value_counts()
    n_grams[:15].plot.barh()
    plt.title(names[n-1])
    plt.tight_layout()
    plt.savefig(f"./ngrams/{names[n-1]}.png")
    plt.close()